In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
import os

files = {
    "Training": 'path/prediction_train.xlsx',
    "Internal Validation": 'path/prediction_Internal_valid.xlsx',
    "Prospective Validation": 'path/prediction_Prospective_valid.xlsx',
    "External Validation": 'path/prediction_External_valid.xlsx',
}

n_bins = 5
save_dir = 'path/'

def compute_ece(y_true, y_prob, n_bins=5):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1

    ece = 0.0
    for i in range(n_bins):
        mask = bin_ids == i
        if np.any(mask):
            acc = np.mean(y_true[mask])
            conf = np.mean(y_prob[mask])
            ece += np.abs(acc - conf) * np.sum(mask) / len(y_true)
    return ece

plt.figure(facecolor='white', figsize=(5, 5))

colors = {"Training": "#6A51A3", "Internal Validation": "#4DB6AC","Prospective Validation": "#4F81BD", "External Validation": "#FF9F1C"}

for name, file_path in files.items():
    df = pd.read_excel(file_path)

    y_true = df.iloc[:, 0].values
    y_prob = df.iloc[:, 2].values

    mask = ~np.isnan(y_true) & ~np.isnan(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="quantile")

    ece = compute_ece(y_true, y_prob, n_bins=n_bins)

    plt.plot(prob_pred, prob_true, marker='o', markersize=3, linewidth=1.5, color=colors[name], label=f"{name} (ECE = {ece:.3f})")

plt.plot([0, 1], [0, 1], linestyle='--', color='black')
plt.xlabel("Predicted probability", fontsize=12)
plt.ylabel("True probability", fontsize=12)
plt.gca().set_aspect('equal', adjustable='box')
plt.xlim([-0.03, 1.03])
plt.ylim([-0.03, 1.03])
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
ax = plt.gca()
ax.spines['top'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.tick_params(axis='both', width=1)
plt.legend(fontsize=9, loc='upper left', frameon=False, labelspacing=1, handlelength=2)

plt.savefig(os.path.join(save_dir, 'Calibration_curves.pdf'), bbox_inches='tight', dpi=600, transparent=True)
plt.show()